In [145]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.pipeline import Pipeline 
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

In [147]:
df = pd.read_csv('steam_games_cleaned.csv')

**Feature ENGG**

In [150]:
current_year = 2026
df['game_age'] = current_year - df['release_year']

A new feature game_age was created to represent how long a game has been available since its release. Older games have had more time to accumulate user reviews, which may influence their popularity. This feature is calculated as the difference between the current year and the game's release year.

In [153]:
df.drop(columns=['release_year'], inplace=True)

In [155]:
df['log_review_count'] = np.log1p(df['review_count'])

In [157]:

df['genres'] = df['genres'].fillna('Unknown')

df['genre_count'] = df['genres'].apply(lambda x: len(x.split(',')))

The genre_count feature was introduced to capture the number of genres associated with each game. Games belonging to multiple genres may appeal to a broader audience, which could potentially increase player engagement and the number of reviews.

In [160]:
# keep top 20 developers
top_dev = df['developer'].value_counts().nlargest(20).index
df['developer'] = df['developer'].apply(lambda x: x if x in top_dev else 'Other')

# keep top 20 publishers
top_pub = df['publisher'].value_counts().nlargest(20).index
df['publisher'] = df['publisher'].apply(lambda x: x if x in top_pub else 'Other')

In [162]:
df['price_bucket'] = pd.cut(
    df['price'],
    bins=[-1,0,500,1500,3000,10000],
    labels=['Free','Cheap','Mid','Expensive','Premium']
)

In [164]:
df['score_bucket'] = pd.cut(
    df['review_score'],
    bins=[0,50,70,85,100],
    labels=['Bad','Average','Good','Excellent']
)

In [166]:
df['release_season'] = (df['release_month'] % 12) // 3 + 1

In [169]:
df.head(5)

,game_name,price,review_count,developer,publisher,genres,release_month,review_sentiment,review_score,game_age,log_review_count,genre_count,price_bucket,score_bucket,release_season
0,Slay the Spire 2,1100.0,13496,Other,Other,"Indie,Strategy,Early Access",3,Overwhelmingly Positive,97,0,9.510223,3,Mid,Excellent,2
1,Resident Evil Requiem,4399.0,34291,"CAPCOM Co., Ltd.","CAPCOM Co., Ltd.","Action,Adventure",2,Overwhelmingly Positive,96,0,10.442667,2,Premium,Excellent,1
2,Counter-Strike 2,0.0,2496132,Valve,Other,"Action,Free To Play",8,Very Positive,86,14,14.730253,2,Free,Excellent,3
3,Marathon,2449.0,12119,Other,Other,Action,3,Very Positive,90,0,9.402612,1,Expensive,Excellent,2
4,ARC Raiders,2467.0,175346,Other,Other,Action,10,Very Positive,86,1,12.074522,1,Expensive,Excellent,4


In [171]:
num_cols = ['price','review_count','release_year','release_month','review_score']
cat_cols = ['developer','publisher','genres','review_sentiment']

In [173]:
for cols in cat_cols :
    print(f"{cols} -> {df[cols].unique()}")

developer -> ['Other' 'CAPCOM Co., Ltd.' 'Valve' 'Ubisoft Montreal' 'Ubisoft'
 'RocketWerkz' 'Maxis' 'FromSoftware, Inc.' 'Bethesda Game Studios'
 'SCS Software' 'KONAMI' 'Saber Interactive' 'Square Enix'
 'Ludeon Studios' 'Paradox Development Studio'
 'KOEI TECMO GAMES CO., LTD.' 'Massive Monster' 'Frontier Developments'
 'CREATIVE ASSEMBLY, Feral Interactive' 'Ryu Ga Gotoku Studio']
publisher -> ['Other' 'CAPCOM Co., Ltd.' 'Electronic Arts' 'Ubisoft'
 'PlayStation Publishing LLC' '2K' 'Paradox Interactive'
 'Coffee Stain Publishing' 'Bethesda Softworks' 'SCS Software'
 'Devolver Digital' 'Focus Entertainment' 'Square Enix'
 'Xbox Game Studios' 'Hooded Horse' 'Activision' 'SEGA' 'Team17'
 'Bandai Namco Entertainment' 'Nacon']
genres -> ['Indie,Strategy,Early Access' 'Action,Adventure' 'Action,Free To Play'
 'Action' 'Action,Adventure,Free To Play' 'Action,Adventure,Racing'
 'Action,Adventure,Massively Multiplayer' 'Action,RPG,Free To Play'
 'Action,Adventure,RPG' 'Simulation,Sports' '

In [175]:
genre_dummies = df['genres'].str.get_dummies(',')

df = pd.concat([df, genre_dummies], axis=1)

df.drop(columns=['genres'], inplace=True)

In [177]:
df

,game_name,price,review_count,developer,publisher,release_month,review_sentiment,review_score,game_age,log_review_count,...,Early Access,Free To Play,Indie,Massively Multiplayer,RPG,Racing,Simulation,Sports,Strategy,Unknown
0,Slay the Spire 2,1100.0,13496,Other,Other,3,Overwhelmingly Positive,97,0,9.510223,...,1,0,1,0,0,0,0,0,1,0
1,Resident Evil Requiem,4399.0,34291,"CAPCOM Co., Ltd.","CAPCOM Co., Ltd.",2,Overwhelmingly Positive,96,0,10.442667,...,0,0,0,0,0,0,0,0,0,0
2,Counter-Strike 2,0.0,2496132,Valve,Other,8,Very Positive,86,14,14.730253,...,0,1,0,0,0,0,0,0,0,0
3,Marathon,2449.0,12119,Other,Other,3,Very Positive,90,0,9.402612,...,0,0,0,0,0,0,0,0,0,0
4,ARC Raiders,2467.0,175346,Other,Other,10,Very Positive,86,1,12.074522,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1959,Foxhole - Supporters Edition,880.0,209,Other,Other,2,Very Positive,89,0,5.347108,...,0,0,1,1,0,0,0,0,0,0
1960,HELLCARD,1100.0,1630,Other,Other,2,Very Positive,88,2,7.396949,...,0,0,1,0,1,0,0,0,0,0
1961,Total War: EMPIRE – Definitive Edition,1499.0,12055,Other,SEGA,3,Very Positive,89,17,9.397318,...,0,0,0,0,0,0,0,0,1,0
1962,Dawncaster | The RPG Cardventure,880.0,172,Other,Other,2,Very Positive,82,0,5.153292,...,0,0,1,0,1,0,0,0,1,0


In [179]:
df = df.drop(columns = ['game_name'] )

**Spliting Data**

In [183]:
x= df.drop(columns = ['review_count','log_review_count'])
y = df['log_review_count']

In [185]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size = 0.2,random_state=42)


**Encoding**

In [188]:

pipe = Pipeline([
    ('preprocess', encode),
    ('model', GradientBoostingRegressor(random_state=42))
])

In [190]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

In [196]:
x_train

,price,developer,publisher,release_month,review_sentiment,review_score,game_age,genre_count,price_bucket,score_bucket,...,Early Access,Free To Play,Indie,Massively Multiplayer,RPG,Racing,Simulation,Sports,Strategy,Unknown
910,0.0,Other,Other,12,Mostly Positive,72,3,3,Free,Good,...,0,0,0,0,1,0,0,0,0,0
1427,570.0,Other,Other,10,Overwhelmingly Positive,95,3,3,Mid,Excellent,...,1,0,0,0,1,0,0,0,0,0
988,579.0,Other,Other,4,Overwhelmingly Positive,98,1,1,Mid,Excellent,...,1,0,0,0,0,0,0,0,0,0
1829,1859.0,Other,Paradox Interactive,2,Mixed,45,4,3,Expensive,Bad,...,0,0,0,0,1,0,1,0,1,0
1765,0.0,Other,Other,10,Very Positive,90,1,2,Free,Excellent,...,0,1,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1130,1100.0,Other,Nacon,3,Very Positive,92,3,3,Mid,Excellent,...,0,0,1,0,1,0,0,0,0,0
1294,0.0,Other,Other,10,Mixed,53,4,3,Free,Average,...,0,0,0,0,1,0,0,0,0,0
860,0.0,Other,Other,11,Mixed,59,9,3,Free,Average,...,0,1,0,1,1,0,0,0,0,0
1459,880.0,Other,Other,7,Overwhelmingly Positive,95,5,2,Mid,Excellent,...,0,0,1,0,0,0,0,0,0,0


In [223]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor
    
    pipe = Pipeline([
        ('preprocess', encode),
        ('model', GradientBoostingRegressor(random_state=42))
    ])

In [225]:
param_grid = {
    
    'model__n_estimators': [200, 400, 600, 800],
    
    'model__learning_rate': [0.01, 0.03, 0.05, 0.1],
    
    'model__max_depth': [3, 4, 5, 6],
    
    'model__min_samples_split': [2, 5, 10],
    
    'model__min_samples_leaf': [1, 3, 5],
    
    'model__subsample': [0.7, 0.8, 0.9, 1.0]
}

In [227]:
from sklearn.model_selection import RandomizedSearchCV

search = RandomizedSearchCV(
    pipe,
    param_distributions=param_grid,
    n_iter=30,
    scoring='r2',
    cv=5,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

In [229]:
search.fit(x_train, y_train)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


RandomizedSearchCV(cv=5,
                   estimator=Pipeline(steps=[('preprocess',
                                              ColumnTransformer(remainder='passthrough',
                                                                transformers=[('ord',
                                                                               OrdinalEncoder(categories=[['Very '
                                                                                                           'Negative',
                                                                                                           'Mostly '
                                                                                                           'Negative',
                                                                                                           'Mixed',
                                                                                                           'Positive',
                                                                                                           'Mostly '
                                                                                                           'Positive',
                                                                                                           'Very '
                                                                                                           'Positive',
                                                                                                           'Overwhelmingly '
                                                                                                           'Positive']]),
                                                                               ['review_sentiment']),
                                                                              ('ohe',
                                                                               OneHotEncoder(handle_unknown='ignore'),
                                                                               ['deve...
                                             ('model',
                                              GradientBoostingRegressor(random_state=42))]),
                   n_iter=30, n_jobs=-1,
                   param_distributions={'model__learning_rate': [0.01, 0.03,
                                                                 0.05, 0.1],
                                        'model__max_depth': [3, 4, 5, 6],
                                        'model__min_samples_leaf': [1, 3, 5],
                                        'model__min_samples_split': [2, 5, 10],
                                        'model__n_estimators': [200, 400, 600,
                                                                800],
                                        'model__subsample': [0.7, 0.8, 0.9,
                                                             1.0]},
                   random_state=42, scoring='r2', verbose=1)

In [230]:
print("Best Parameters:")
print(search.best_params_)

Best Parameters:
{'model__subsample': 0.7, 'model__n_estimators': 800, 'model__min_samples_split': 5, 'model__min_samples_leaf': 5, 'model__max_depth': 3, 'model__learning_rate': 0.03}


In [231]:
print("Best CV Score:", search.best_score_)

Best CV Score: 0.5230726571327048


In [238]:
best_model = search.best_estimator_

y_pred = best_model.predict(x_test)

from sklearn.metrics import r2_score

print("Test R2:", r2_score(y_test, y_pred))

Test R2: 0.5531652375343146


In [ ]:
from lightgbm import LGBMClassifier

model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=-1,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

from sklearn.metrics import accuracy_score
print(accuracy_score(y_test, pred))